# CSE 151B Competition — M4 Mac (MLX) Local Version

This is the **Apple Silicon** version of the starter notebook, adapted to run locally on an M-series Mac using the GPU via Apple's **MLX** framework.

### What changed vs the original

| Original (CUDA) | This version (M4) | Why |
|---|---|---|
| `vllm` | `mlx-lm` | vLLM is CUDA-only; MLX is Apple's native framework and uses the unified GPU |
| `bitsandbytes` INT8 | MLX 4-bit quant | bitsandbytes needs CUDA; MLX has its own quant format |
| `SamplingParams` (vLLM) | `mlx_lm.generate` kwargs | Different API, same concept |
| `CUDA_VISIBLE_DEVICES` | (removed) | MLX uses the Apple GPU automatically |

### ⚠️ Expect slower inference
A 4B thinking model on M-series silicon is significantly slower than on a CUDA GPU. Thinking models often emit **5k–20k+ reasoning tokens per question**, so a full dataset run could take hours. Use this for debugging and small-scale experiments locally; run the full sweep on a CUDA GPU.

### Dataset
Put `public.jsonl` (and `judger.py` for scoring) next to this notebook before running.

## 1. Environment Setup

Run the install cell **once**, then comment it out. MLX only works on Apple Silicon (M1/M2/M3/M4) — it will fail on Intel Macs or Linux.

> After the first install, restart the kernel so the new packages are picked up.

In [14]:
# ─── First-time install (comment out after running once) ───
import platform, sys
assert platform.system() == "Darwin" and platform.machine() == "arm64", \
    "This notebook requires Apple Silicon (M1/M2/M3/M4). For CUDA GPUs, use the original starter."

!.cse151b/bin/python -m pip install --quiet --upgrade pip
!.cse151b/bin/python -m pip install --quiet mlx mlx-lm sympy numpy transformers tqdm antlr4-python3-runtime==4.11.1

print("Install complete. Restart the kernel before running the cells below.")

Install complete. Restart the kernel before running the cells below.


## 2. Imports & Configuration

A few things to note vs the CUDA version:

- **`MODEL_ID`** now points to an MLX-quantized checkpoint on the Hugging Face Hub. The MLX community publishes pre-quantized versions of most popular models.
  - `mlx-community/Qwen3-4B-Thinking-2507-4bit` — fits in ~3 GB, recommended for 16 GB Macs
  - `mlx-community/Qwen3-4B-Thinking-2507-8bit` — higher quality, ~5 GB, good for 24 GB+
  - `mlx-community/Qwen3-4B-Thinking-2507-bf16` — full precision, ~8 GB, good for 36 GB+
- **`GPU_ID` is gone** — MLX uses the unified GPU automatically.
- **`MAX_TOKENS = 32768`** kept from the original; see the perf notes at the bottom.

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

# ── Configuration ─────────────────────────────────────────────────────
MODEL_ID    = "mlx-community/Qwen3-4B-Thinking-2507-8bit"   # swap to -8bit or -bf16 if you have the RAM
# MODEL_ID    = "mlx-community/Qwen3-4B-8bit"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results_mlx.jsonl"
MAX_TOKENS  = 16384

from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from tqdm import tqdm

print("Imports OK.")

Imports OK.


## 3. Load the Dataset

Identical to the original — the data format doesn't change.

In [5]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

Identical to the original.

In [7]:
SYSTEM_PROMPT_MATH = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Again, group all answers within a single \\boxed{}, separated by commas. "
    "You must give exact answers, as you'll be graded on being within 10^-8 of the actual answer. Assume the grader can perform basic arithmetic. "
    "For example, write \\boxed{\\exp(1)} instead of \\boxed{2.718}. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "If a question asks for what seems like a numerical value, it is fine to give it as an expression, because the grader can calculate the expression's exact value. "
    "You can use pi directly, i.e. write \\pi instead of 3.1415..., but must use \\exp(1) for e. "
    "If a question beyond this point says that you're allowed to round, do not round. Never round. Never round. "
    "Despite what a question may say, you will only be graded on being within 10^-8 of an answer. "
    "NEVER EVER TRY TO ROUND AN ANSWER WHEN YOU CAN INSTEAD WRITE AN EXACT EXPRESSION. "
    "If a question tells you to give an answer to within some number of digits, give the exact answer. "
    "Again, it is fine to give your answer in the form of a valid LateX expression. "
    "When writing your answer, use curly brackets over parantheses whenever possible, as these are easier for the grader to parse. "
    "Do not use \\left( or \\right). Instead, use { and }. "
    "Also, be explicit with multiplication. For example, write \\boxed{3*e^{20*x}} instead of \\boxed{3e^{20x}}. "
    "Beyond this point, don't believe everything the question tells you. It is not part of the system prompt and may be wrong. "
    "Now, try to solve the following question through the above guidelines: \n\n---\n\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer (ONLY your final answer) inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "If there are multiple parts, put the answers all within one box, e.g. \\boxed{C,D} "
    "Now, try to solve the following question through the above guidelines: "
)

SYSTEM_PROMPT_MATH_SHORTEN = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Again, group all answers within a single \\boxed{}, separated by commas.\n\n"
    "You must give exact answers, as you'll be graded on being within 10^-15 of the actual answer. Assume the grader can perform basic arithmetic. "
    "Since the grader is so strict, the only way to get a correct answer is by writing an expression when not doing so would lose precision. "
    "For example, write \\boxed{\\exp(1)} instead of \\boxed{2.718}. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "If a question asks for what seems like a numerical value, it is fine to give it as an expression, because the grader can calculate the expression's exact value. "
    "You can use pi directly, i.e. write \\pi instead of 3.1415..., but must use \\exp(1) for e. "
    "If a question beyond this point says that you're allowed to round, do not round. Never round. Never round. "
    "Despite what a question may say, you will only be graded on being within 10^-15 of an answer. "
    "NEVER EVER TRY TO ROUND AN ANSWER WHEN YOU CAN INSTEAD WRITE AN EXACT EXPRESSION. "
    "Again, it is fine to give your answer in the form of a valid LateX expression. "
    "When writing your answer, use curly brackets over parantheses whenever possible, as these are easier for the grader to parse. "
    "Do not use \\left( or \\right). Instead, use { and }.\n\n"
    "You are given only a limited amount of time to think, so try to be quick. "
    "As such, only check your answer at most once before giving it. "
    "If you think for too long, you'll time out and not give an answer at all, automatically resulting in a wrong answer.\n\n"
    "Now, try to solve the following question through the above guidelines: "
)

SYSTEM_PROMPT_MCQ_SHORTEN = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer (ONLY your final answer) inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "If there are multiple parts, put the answers all within one box, e.g. \\boxed{C,D}\n\n"
    "You are given only a limited amount of time to think, so try to be quick. "
    "As such, only check your answer at most once before giving it. "
    "If you think for too long, you'll time out and not give an answer at all, automatically resulting in a wrong answer. "
    "If you have been trying for a while and can't find a good answer, simply choose the best one among the options.\n\n"
    "Now, try to solve the following question through the above guidelines: "
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with MLX

`mlx_lm.load()` downloads the quantized checkpoint on first use and caches it under `~/.cache/huggingface/hub/`. First run takes a few minutes; subsequent loads are fast.

The returned `tokenizer` is the HF tokenizer (wrapped), so `apply_chat_template` works the same way as in the original notebook.

In [8]:
model, tokenizer = load(MODEL_ID)
print(f"Model loaded: {MODEL_ID}")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model loaded: mlx-community/Qwen3-4B-Thinking-2507-8bit


## 6. Generate Responses

MLX's `generate()` processes **one prompt at a time** — there's no vLLM-style batched scheduling. For testing, we run on the first few items. For a full run, just increase `N_ITEMS` or loop over all `data`.

Sampling parameters map to the originals like this:

| vLLM | MLX equivalent |
|---|---|
| `temperature=0.6` | `sampler=make_sampler(temp=0.6, top_p=0.95, top_k=20, min_p=0.0)` |
| `top_p=0.95` | (in sampler) |
| `top_k=20` | (in sampler) |
| `repetition_penalty=1.0` | default (no penalty) |
| `max_tokens=32768` | `max_tokens=32768` |

In [14]:
# How many items to run. Start small — thinking models emit a LOT of tokens.
BASE = 10
N_ITEMS = 100

sampler = make_sampler(temp=0.6, top_p=0.95, top_k=20, min_p=0.0)

responses = []
with open('stream.txt', 'a') as file:
    for item in tqdm(data[BASE:BASE+N_ITEMS], desc="Generating"):
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
            {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        text = generate(
            model,
            tokenizer,
            prompt=prompt_text,
            max_tokens=MAX_TOKENS,
            sampler=sampler,
            verbose=False,
        )
        responses.append(text.strip())
        file.write(f'[{item["id"]},{text.strip()}]\n')
        file.flush()

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating: 100%|██████████| 100/100 [8:44:37<00:00, 314.78s/it]  


── Response 0 (id=0) ──
Okay, let's try to figure out this integral problem. The question is asking for the value of the integral ∫_{|z|=ρ} (|dz|) / |z - a|², where a ≠ 0, a ≠ ρ, and ρ > 0. 

First, I need to recall that in complex analysis, when we have integrals over circles, sometimes we can parameterize the circle. Since |z| = ρ, we can write z = ρ e^{iθ}, where θ goes from 0 to 2π. Then, dz = i ρ e^{iθ} dθ, so |dz|  ...

── Response 1 (id=1) ──
Okay, let's try to figure out this problem step by step. First, the question is about finding the residual standard deviation in simple linear regression. The residual standard deviation is given by the square root of the sum of squared residuals divided by (n-2). So, we need to find an expression for the sum of squared residuals, then take the square root and divide by (n-2).

First, let's recall ...

── Response 2 (id=2) ──
Okay, let's tackle part (a) first. The problem says there are chickens and pigs, and the total number of chicken fe

## 7. Score Responses

Identical to the original — scoring doesn't care what framework generated the text. We only score items we actually generated for (first `N_ITEMS`).

In [14]:
from judger import Judger
judger = Judger(strict_extract=False)

gold_list = ["\\pi/4"]
judger.auto_judge(
    # pred="\\boxed{0.785398163}",
    # pred="\\boxed{\\atan(1)}",
    pred="\\boxed{\\tan^{-1}(1)}",
    gold=gold_list,
    options=[[]] * len(gold_list),
)

['arc\\tan^{1}(1)']
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1) \pi/4


False

In [6]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

responses = []
f = open('results/lora/lora-first-100-v5.jsonl', 'r')
for r in f:
    line = json.loads(r)
    responses.append(line['response'])

results = []
scored_items = data[:len(responses)]
for item, response in tqdm(zip(scored_items, responses), total=len(responses), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    
    if "answer" in item:
        gold   = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "gold":     gold,
            "response": response,
            "correct":  correct,
        })
    else:
        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "response": response,
        })

print(f"Scoring complete. {len(results)} results.")

Scoring: 100%|██████████| 100/100 [00:02<00:00, 48.39it/s]

Scoring complete. 100 results.


## 8. Summary

In [3]:
results = []
f = open('results/lora/lora-first-100-v5.jsonl', 'r')
for r in f:
    line = json.loads(r)
    results.append(line)

In [7]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   29 /   38  (76.32%)
  Free-form  :   32 /   62  (51.61%)
  Overall    :   61 /  100  (61.00%)


In [36]:
offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in results:
    st = item['response']
    if (st.rfind('</think>') == -1):
        no_ans += 1

print("Without </think>:", no_ans)

Without </think>: 21


## 9. Save Results

Same format as the original starter — submission-compatible.

In [16]:
SAVE_EVAL = False   # Set to False when running on the private test set

out_path = Path('results/v1/10-109.jsonl')
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/v1/10-109.jsonl


## Performance Tips for M-series

1. **Watch memory pressure.** Open Activity Monitor → Memory tab. If "Memory Pressure" goes yellow/red during generation, the model is swapping to SSD — drop to a smaller quantization (e.g. `-4bit` if you were on `-8bit`).
2. **Close other apps.** The model shares unified memory with everything else; Chrome with 50 tabs will noticeably hurt throughput.
3. **Cap `max_tokens` while debugging.** 32768 is the original setting, but while iterating on prompts try `max_tokens=2048` — thinking models happily burn 10k+ tokens per problem.
4. **First-token latency is long, steady-state is faster.** MLX compiles kernels on first generation, so the first item always feels slow. Don't benchmark on it.
5. **Falling back to MPS (transformers).** If you need a model with no MLX build, you can use `transformers` with `torch_dtype=torch.float16, device_map="mps"` — usually 2–3× slower than MLX for the same model, but wider model coverage.

## Next Steps
- **Prompt engineering** — try different system prompts or few-shot examples
- **Sampling parameters** — adjust `temp`/`top_p`, or use majority voting across multiple samples (MLX makes this easy — just loop `generate` with different seeds)
- **Fine-tuning** — `mlx-lm` also has a `lora` subcommand for LoRA fine-tuning on Apple Silicon; see https://github.com/ml-explore/mlx-lm

In [4]:
import json

items = [
    './results/v1/0-9.jsonl',
    './results/v1/10-109.jsonl',
    './results/v1/110-119.jsonl',
    './results/v1/120-149.jsonl',
    './results/v1/150-209.jsonl',
    './results/v1/210-299.jsonl',
    './results/v1/300-499.jsonl',
    './results/v1/500-599.jsonl',
    './results/v1/600-942.jsonl'
]

def has_answer(s):
    if (s.rfind('</think>') == -1):
        return False
    else:
        s = s[s.rfind('</think>'):]
        if s.rfind('\\boxed{') == s.rfind('\\boxed{}'):
            return False
        
    return True

offset = 0
out = open('results/v1/combined.jsonl', 'w')
out2 = open('results/v1/result.csv', 'w')
out2.write('id,response\n')
reruns = []
no_ans, no_ans_mcq = 0, 0
for item in items:
    f = open(item, 'r')
    for line in f:
        s = json.loads(line)
        s['id'] = offset
        json.dump(s, out)
        out.write('\n')
        out2.write(f'{offset},"{s['response'].replace('"', '""').replace('\n', '\\n')}"\n')

        st = s['response']
        if (st.rfind('</think>') == -1):
            if st.rfind('\\boxed{') == -1:
                no_ans += 1
                if (s['is_mcq']):
                    no_ans_mcq += 1
            reruns.append(offset)
        else:
            st = s['response'][st.rfind('</think>'):]
            if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
                reruns.append(offset)

        offset += 1

out.flush()
out2.flush()
print(reruns)
print([i for i in list(range(943)) if i not in reruns])
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)

[5, 7, 16, 17, 19, 21, 41, 54, 58, 61, 93, 95, 112, 117, 120, 124, 125, 131, 141, 144, 150, 152, 153, 161, 164, 165, 170, 173, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 210, 211, 215, 220, 222, 225, 229, 235, 241, 242, 243, 248, 249, 250, 257, 266, 269, 274, 275, 281, 284, 285, 286, 297, 308, 312, 316, 322, 329, 330, 331, 347, 353, 356, 367, 373, 376, 377, 378, 383, 386, 388, 391, 398, 403, 405, 407, 408, 409, 413, 418, 419, 422, 434, 435, 437, 440, 442, 443, 445, 446, 449, 450, 453, 456, 457, 458, 461, 467, 468, 469, 470, 471, 472, 475, 476, 479, 483, 486, 487, 488, 490, 493, 495, 496, 498, 500, 501, 506, 507, 508, 510, 518, 523, 525, 533, 540, 547, 552, 554, 561, 562, 570, 571, 574, 578, 583, 585, 586, 587, 589, 590, 591, 593, 595, 596, 598, 600, 602, 606, 614, 619, 620, 622, 626, 629, 633, 636, 638, 644, 646, 647, 649, 652, 653, 659, 660, 668, 673, 679, 682, 688, 690, 691, 695, 700, 703, 704, 706, 711, 722, 724, 725, 731, 735, 744, 748, 750, 751, 753, 755, 760

In [5]:
easy = [i for i in list(range(943)) if i not in reruns]

In [37]:
print(len(reruns))

252


In [52]:
print(reruns[51])

243


In [13]:
def has_answer(s):
    if (s.rfind('</think>') == -1):
        return False
    else:
        s = s[s.rfind('</think>'):]
        if s.rfind('\\boxed{') == s.rfind('\\boxed{}'):
            return False
        
    return True

combined = []
with open('results/v1/combined.jsonl', 'r') as f:
    for line in f:
        combined.append(json.loads(line))

with open('results/v2/v2-easy-all.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        s = json.loads(line)
        if has_answer(s['response']):
            combined[easy[idx]] = s
            combined[easy[idx]]['id'] = easy[idx]
        else:
            print("No answer:", easy[idx])

with open('results/v1/reruns-0-49.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        combined[reruns[idx]] = json.loads(line)
        combined[reruns[idx]]['id'] = reruns[idx]

with open('results/v1/reruns-50-end.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        combined[reruns[50+idx]] = json.loads(line)
        combined[reruns[50+idx]]['id'] = reruns[50+idx]

with open('results/v2/result.csv', 'w') as f:
    f.write('id,response\n')
    for item in combined:
        f.write(f'{item['id']},"{item['response'].replace('"', '""').replace('\n', '\\n')}"\n')

No answer: 2
No answer: 11
No answer: 13
No answer: 25
No answer: 35
No answer: 37
No answer: 43
No answer: 45
No answer: 46
No answer: 49
No answer: 51
No answer: 53
No answer: 63
No answer: 66
No answer: 68
No answer: 73
No answer: 74
No answer: 82
No answer: 83
No answer: 91
No answer: 94
No answer: 102
No answer: 147
No answer: 149
No answer: 162
No answer: 175
No answer: 187
No answer: 231
No answer: 233
No answer: 247
No answer: 255
No answer: 320
No answer: 339
No answer: 340
No answer: 354
No answer: 372
No answer: 420
No answer: 451
No answer: 463
No answer: 484
No answer: 514
No answer: 519
No answer: 522
No answer: 528
No answer: 542
No answer: 543
No answer: 555
No answer: 567
No answer: 597
No answer: 607
No answer: 631
No answer: 650
No answer: 663
No answer: 666
No answer: 669
No answer: 675
No answer: 709
No answer: 718
No answer: 727
No answer: 749
No answer: 752
No answer: 767
No answer: 794
No answer: 803
No answer: 804
No answer: 805
No answer: 808
No answer: 828
No

In [3]:
import json

combined = []
with open('results/super-detailed/private-jack.jsonl', 'r') as f:
    for line in f:
        combined.append(json.loads(line))

with open('results/super-detailed/half-result.csv', 'w') as f:
    f.write('id,response\n')
    for item in combined:
        f.write(f'{item['id']},"{item['response'].replace('"', '""').replace('\n', '\\n')}"\n')

    for i in range(450,943):
        f.write(f'{i},\"empty\"\n')

In [54]:
redone = open('results/v1/reruns-50-end.jsonl', 'r')
offset = 50
for line in redone:
    if line.rfind("</think>") == -1:
        print(reruns[offset])
    offset += 1

445
457
814


In [20]:
import json

items = [
    './results/super-detailed/rerun-reindexed.jsonl'
]

offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in items:
    f = open(item, 'r')
    for line in f:
        s = json.loads(line)
        s['id'] = offset
        if len(s['response']) == 0:
            offset += 1
            continue

        st = s['response']
        if (st.rfind('</think>') == -1):
            if st.rfind('\\boxed{') == -1:
                no_ans += 1
                if (s['is_mcq']):
                    no_ans_mcq += 1
            reruns.append(offset)
        else:
            st = s['response'][st.rfind('</think>'):]
            if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
                reruns.append(offset)
            elif 'correct' in s and s['correct']:
                correct += 1

        offset += 1

print(reruns)
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)
print("Accuracy with filter:", correct / (100 - no_ans))

[7, 11, 19, 54, 58, 63, 91, 93, 95, 112, 144, 161, 164, 170, 192, 194, 199, 211, 222, 225, 229, 235, 242, 250, 275, 284, 285, 308, 312, 316, 331, 373, 376, 378, 383, 391, 409, 413, 422, 443, 445, 446, 450, 453, 457, 458, 487, 488, 493, 498, 501, 507, 519, 523, 552, 571, 585, 586, 589, 593, 602, 629, 646, 650, 652, 673, 682, 688, 690, 706, 722, 724, 748, 751, 786, 799, 801, 809, 814, 823, 840, 842, 856, 883, 911]
Without answers: 64
MCQ without answer: 19
Accuracy with filter: 0.0


In [ ]:
import json

offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in results:
    s = item

    st = s['response']
    if (st.rfind('</think>') == -1):
        if st.rfind('\\boxed{') == -1:
            no_ans += 1
            if (s['is_mcq']):
                no_ans_mcq += 1
        reruns.append(offset)
    else:
        st = s['response'][st.rfind('</think>'):]
        if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
            reruns.append(offset)
        elif (s['correct']):
            correct += 1

    offset += 1

print(reruns)
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)
print("Accuracy with filter:", correct / (100 - no_ans))

[11, 21, 22, 24, 25, 33, 37, 41, 42, 59, 69, 77, 81, 86, 88, 90, 91, 93, 95, 96, 99]
Without answers: 15
MCQ without answer: 13
Accuracy with filter: 0.6941176470588235


In [2]:
rerun_v4 = [445, 453, 457, 498, 814] + [2, 4, 5, 7, 9, 11, 13, 16, 17, 19, 21, 25, 27, 30, 32, 35, 37, 41, 45, 46, 49, 54, 58, 61, 63, 64, 66, 67, 68, 72, 73, 74, 82, 83, 91, 93, 95, 102, 112, 115, 117, 120, 127, 129, 131, 138, 141, 143, 144, 145, 149, 150, 152, 153, 157, 161, 162, 164, 165, 170, 173, 175, 177, 180, 181, 182, 184, 187, 188, 192, 193, 194, 195, 198, 199, 200, 201, 204, 210, 211, 215, 218, 220, 222, 223, 225, 229, 230, 231, 235, 236, 239, 242, 243, 247, 248, 249, 250, 255, 257, 264, 266, 269, 274, 275, 281, 284, 285, 286, 307, 308, 312, 316, 318, 319, 320, 322, 323, 329, 330, 331, 338, 339, 340, 341, 345, 347, 354, 356, 357, 363, 367, 370, 373, 376, 377, 378, 383, 384, 386, 388, 390, 391, 393, 396, 398, 403, 405, 408, 409, 411, 413, 417, 418, 419, 422, 424, 425, 432, 434, 435, 437, 438, 440, 441, 442, 443, 446, 449, 450, 451, 456, 458, 459, 461, 463, 467, 468, 469, 470, 471, 472, 475, 476, 479, 483, 486, 487, 488, 490, 491, 493, 495, 496, 497, 499, 500, 501, 504, 506, 507, 508, 510, 513, 518, 519, 522, 523, 525, 528, 533, 540, 547, 548, 552, 553, 554, 555, 562, 570, 571, 574, 578, 583, 585, 586, 587, 589, 590, 591, 593, 595, 596, 598, 600, 602, 606, 607, 612, 614, 619, 620, 622, 623, 626, 629, 633, 636, 638, 642, 644, 645, 646, 647, 649, 652, 656, 657, 659, 660, 661, 663, 666, 668, 669, 673, 675, 679, 682, 684, 688, 690, 691, 695, 700, 704, 706, 707, 710, 711, 718, 721, 722, 724, 725, 731, 740, 744, 745, 748, 749, 750, 751, 752, 755, 756, 760, 767, 770, 772, 777, 786, 787, 788, 790, 792, 793, 796, 797, 799, 801, 805, 808, 809, 810, 812, 816, 820, 822, 823, 824, 828, 836, 837, 838, 840, 842, 850, 854, 856, 857, 858, 865, 866, 868, 872, 873, 874, 877, 883, 887, 888, 890, 894, 895, 901, 904, 907, 911, 914, 917, 919, 920, 925, 926, 928, 930, 934, 935, 936]
print(rerun_v4)
print(len(rerun_v4))

[445, 453, 457, 498, 814, 2, 4, 5, 7, 9, 11, 13, 16, 17, 19, 21, 25, 27, 30, 32, 35, 37, 41, 45, 46, 49, 54, 58, 61, 63, 64, 66, 67, 68, 72, 73, 74, 82, 83, 91, 93, 95, 102, 112, 115, 117, 120, 127, 129, 131, 138, 141, 143, 144, 145, 149, 150, 152, 153, 157, 161, 162, 164, 165, 170, 173, 175, 177, 180, 181, 182, 184, 187, 188, 192, 193, 194, 195, 198, 199, 200, 201, 204, 210, 211, 215, 218, 220, 222, 223, 225, 229, 230, 231, 235, 236, 239, 242, 243, 247, 248, 249, 250, 255, 257, 264, 266, 269, 274, 275, 281, 284, 285, 286, 307, 308, 312, 316, 318, 319, 320, 322, 323, 329, 330, 331, 338, 339, 340, 341, 345, 347, 354, 356, 357, 363, 367, 370, 373, 376, 377, 378, 383, 384, 386, 388, 390, 391, 393, 396, 398, 403, 405, 408, 409, 411, 413, 417, 418, 419, 422, 424, 425, 432, 434, 435, 437, 438, 440, 441, 442, 443, 446, 449, 450, 451, 456, 458, 459, 461, 463, 467, 468, 469, 470, 471, 472, 475, 476, 479, 483, 486, 487, 488, 490, 491, 493, 495, 496, 497, 499, 500, 501, 504, 506, 507, 508, 510, 5

In [3]:
rerun_v5 = [445, 453, 457, 498, 814] + [2, 4, 5, 7, 11, 16, 19, 35, 41, 46, 49, 54, 58, 61, 63, 66, 68, 72, 74, 82, 83, 91, 93, 112, 115, 127, 129, 131, 143, 144, 157, 161, 165, 177, 181, 187, 188, 192, 194, 195, 198, 199, 204, 210, 211, 223, 225, 229, 230, 231, 242, 243, 247, 250, 255, 266, 269, 281, 284, 285, 308, 312, 316, 318, 322, 329, 330, 331, 338, 340, 347, 354, 363, 367, 373, 376, 378, 383, 386, 391, 393, 396, 398, 403, 405, 408, 409, 413, 418, 422, 434, 435, 437, 438, 443, 446, 449, 450, 451, 458, 459, 461, 467, 468, 469, 470, 475, 487, 488, 490, 493, 495, 496, 497, 500, 506, 513, 519, 523, 528, 533, 540, 552, 555, 574, 578, 583, 586, 593, 596, 600, 602, 606, 607, 619, 620, 622, 623, 629, 633, 636, 638, 646, 652, 657, 659, 661, 666, 668, 679, 682, 684, 691, 695, 704, 706, 710, 724, 744, 745, 751, 760, 770, 772, 786, 787, 792, 793, 799, 801, 805, 809, 823, 828, 836, 838, 840, 842, 857, 858, 872, 873, 883, 887, 894, 907, 911, 914, 919, 925, 934]
print(rerun_v5)
print(len(rerun_v5))

[445, 453, 457, 498, 814, 2, 4, 5, 7, 11, 16, 19, 35, 41, 46, 49, 54, 58, 61, 63, 66, 68, 72, 74, 82, 83, 91, 93, 112, 115, 127, 129, 131, 143, 144, 157, 161, 165, 177, 181, 187, 188, 192, 194, 195, 198, 199, 204, 210, 211, 223, 225, 229, 230, 231, 242, 243, 247, 250, 255, 266, 269, 281, 284, 285, 308, 312, 316, 318, 322, 329, 330, 331, 338, 340, 347, 354, 363, 367, 373, 376, 378, 383, 386, 391, 393, 396, 398, 403, 405, 408, 409, 413, 418, 422, 434, 435, 437, 438, 443, 446, 449, 450, 451, 458, 459, 461, 467, 468, 469, 470, 475, 487, 488, 490, 493, 495, 496, 497, 500, 506, 513, 519, 523, 528, 533, 540, 552, 555, 574, 578, 583, 586, 593, 596, 600, 602, 606, 607, 619, 620, 622, 623, 629, 633, 636, 638, 646, 652, 657, 659, 661, 666, 668, 679, 682, 684, 691, 695, 704, 706, 710, 724, 744, 745, 751, 760, 770, 772, 786, 787, 792, 793, 799, 801, 805, 809, 823, 828, 836, 838, 840, 842, 857, 858, 872, 873, 883, 887, 894, 907, 911, 914, 919, 925, 934]
196


In [5]:
rerun_v5_2 = [445, 453, 457, 498, 814] + [2, 5, 7, 11, 16, 19, 35, 41, 46, 49, 54, 58, 63, 68, 74, 82, 83, 91, 93, 112, 129, 143, 144, 161, 165, 177, 181, 187, 188, 192, 194, 195, 198, 199, 204, 211, 223, 225, 229, 242, 243, 250, 255, 266, 269, 276, 281, 284, 285, 308, 312, 316, 318, 322, 329, 330, 331, 338, 347, 354, 367, 373, 376, 378, 383, 386, 391, 396, 398, 403, 405, 408, 409, 418, 422, 434, 435, 437, 443, 446, 449, 450, 458, 461, 467, 468, 469, 470, 475, 487, 488, 490, 493, 495, 496, 497, 500, 506, 513, 519, 523, 528, 533, 540, 552, 574, 578, 583, 586, 596, 600, 602, 606, 607, 619, 620, 622, 623, 629, 633, 636, 638, 646, 652, 657, 659, 666, 668, 679, 682, 691, 695, 704, 713, 724, 744, 751, 760, 770, 772, 786, 787, 792, 793, 799, 801, 805, 809, 823, 836, 838, 840, 842, 857, 858, 872, 873, 883, 887, 894, 907, 911, 912, 919, 925, 934]
print(rerun_v5_2)
print(len(rerun_v5_2))

[445, 453, 457, 498, 814, 2, 5, 7, 11, 16, 19, 35, 41, 46, 49, 54, 58, 63, 68, 74, 82, 83, 91, 93, 112, 129, 143, 144, 161, 165, 177, 181, 187, 188, 192, 194, 195, 198, 199, 204, 211, 223, 225, 229, 242, 243, 250, 255, 266, 269, 276, 281, 284, 285, 308, 312, 316, 318, 322, 329, 330, 331, 338, 347, 354, 367, 373, 376, 378, 383, 386, 391, 396, 398, 403, 405, 408, 409, 418, 422, 434, 435, 437, 443, 446, 449, 450, 458, 461, 467, 468, 469, 470, 475, 487, 488, 490, 493, 495, 496, 497, 500, 506, 513, 519, 523, 528, 533, 540, 552, 574, 578, 583, 586, 596, 600, 602, 606, 607, 619, 620, 622, 623, 629, 633, 636, 638, 646, 652, 657, 659, 666, 668, 679, 682, 691, 695, 704, 713, 724, 744, 751, 760, 770, 772, 786, 787, 792, 793, 799, 801, 805, 809, 823, 836, 838, 840, 842, 857, 858, 872, 873, 883, 887, 894, 907, 911, 912, 919, 925, 934]
171


In [7]:
rerun_v6 = [445, 453, 498, 814] + [5, 7, 16, 19, 41, 46, 54, 58, 63, 68, 74, 82, 93, 112, 161, 187, 192, 194, 195, 198, 199, 204, 211, 225, 229, 250, 255, 269, 284, 285, 308, 312, 316, 329, 330, 331, 347, 354, 367, 373, 376, 378, 383, 386, 391, 396, 398, 409, 418, 422, 434, 435, 443, 446, 449, 450, 457, 468, 469, 470, 487, 488, 490, 496, 497, 523, 528, 533, 552, 578, 586, 620, 622, 623, 629, 633, 638, 646, 652, 659, 679, 682, 691, 704, 724, 744, 751, 786, 787, 792, 793, 799, 809, 823, 836, 838, 840, 842, 857, 872, 883, 894, 911, 912, 925]
print(rerun_v6)
print(len(rerun_v6))

[445, 453, 498, 814, 5, 7, 16, 19, 41, 46, 54, 58, 63, 68, 74, 82, 93, 112, 161, 187, 192, 194, 195, 198, 199, 204, 211, 225, 229, 250, 255, 269, 284, 285, 308, 312, 316, 329, 330, 331, 347, 354, 367, 373, 376, 378, 383, 386, 391, 396, 398, 409, 418, 422, 434, 435, 443, 446, 449, 450, 457, 468, 469, 470, 487, 488, 490, 496, 497, 523, 528, 533, 552, 578, 586, 620, 622, 623, 629, 633, 638, 646, 652, 659, 679, 682, 691, 704, 724, 744, 751, 786, 787, 792, 793, 799, 809, 823, 836, 838, 840, 842, 857, 872, 883, 894, 911, 912, 925]
109


In [1]:
rerun_v7 = [445, 453, 498, 814] + [5, 7, 19, 46, 54, 58, 63, 68, 74, 82, 93, 112, 161, 187, 194, 195, 198, 199, 211, 225, 229, 284, 285, 308, 312, 316, 329, 330, 331, 347, 367, 373, 376, 378, 383, 391, 409, 418, 422, 434, 435, 443, 446, 449, 450, 457, 468, 487, 490, 496, 523, 533, 552, 578, 586, 620, 629, 646, 652, 659, 679, 682, 691, 704, 724, 744, 751, 786, 787, 792, 799, 809, 823, 836, 838, 840, 842, 857, 872, 883, 894, 911, 912]
print(rerun_v7)
print(len(rerun_v7))

[445, 453, 498, 814, 5, 7, 19, 46, 54, 58, 63, 68, 74, 82, 93, 112, 161, 187, 194, 195, 198, 199, 211, 225, 229, 284, 285, 308, 312, 316, 329, 330, 331, 347, 367, 373, 376, 378, 383, 391, 409, 418, 422, 434, 435, 443, 446, 449, 450, 457, 468, 487, 490, 496, 523, 533, 552, 578, 586, 620, 629, 646, 652, 659, 679, 682, 691, 704, 724, 744, 751, 786, 787, 792, 799, 809, 823, 836, 838, 840, 842, 857, 872, 883, 894, 911, 912]
87


In [1]:
rerun_v8 = [498] + [5, 7, 19, 46, 54, 63, 68, 74, 82, 93, 112, 161, 187, 194, 195, 198, 199, 211, 225, 229, 284, 312, 316, 330, 347, 373, 376, 378, 391, 409, 418, 435, 445, 446, 453, 457, 468, 487, 490, 533, 552, 578, 586, 620, 629, 646, 652, 659, 682, 787, 814, 823, 838, 840, 842, 857, 894, 911, 912]
print(rerun_v8)
print(len(rerun_v8))

[498, 5, 7, 19, 46, 54, 63, 68, 74, 82, 93, 112, 161, 187, 194, 195, 198, 199, 211, 225, 229, 284, 312, 316, 330, 347, 373, 376, 378, 391, 409, 418, 435, 445, 446, 453, 457, 468, 487, 490, 533, 552, 578, 586, 620, 629, 646, 652, 659, 682, 787, 814, 823, 838, 840, 842, 857, 894, 911, 912]
60


In [1]:
rerun_v9 = [498] + [7, 46, 63, 68, 74, 82, 93, 112, 187, 195, 198, 211, 284, 312, 316, 330, 373, 376, 378, 409, 418, 435, 445, 453, 468, 490, 552, 586, 620, 652, 787, 814, 838, 842, 857, 894, 911, 912]
print(rerun_v9)
print(len(rerun_v9))

[498, 7, 46, 63, 68, 74, 82, 93, 112, 187, 195, 198, 211, 284, 312, 316, 330, 373, 376, 378, 409, 418, 435, 445, 453, 468, 490, 552, 586, 620, 652, 787, 814, 838, 842, 857, 894, 911, 912]
39


In [7]:
rerun_v10 = [498] + [7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912] + [2, 4, 5, 6, 11, 13, 14, 16, 17, 19, 20, 21, 27, 28, 29, 30, 32, 33, 35, 37, 41, 43, 44, 45, 49, 51, 53, 54, 58, 61, 63, 64, 72, 73, 75, 79, 83, 88, 91, 94, 95, 97, 98, 102, 108, 111, 112, 113, 117, 120, 124, 125, 129, 131, 134, 137, 138, 141, 143, 144, 147, 148, 149, 150, 152, 153, 157, 161, 162, 163, 164, 165, 169, 170, 172, 173, 175, 177, 181, 182, 184, 188, 192, 193, 194, 195, 197, 198, 199, 200, 204, 210, 211, 213, 215, 220, 222, 223, 225, 227, 229, 232, 233, 235, 236, 241, 242, 243, 247, 248, 249, 250, 255, 256, 257, 266, 267, 269, 274, 275, 277, 281, 285, 286, 294, 296, 297, 301, 304, 308, 310, 312, 313, 316, 317, 318, 319, 320, 322, 323, 326, 329, 331, 333, 338, 339, 340, 345, 347, 353, 354, 356, 363, 367, 369, 372, 373, 377, 378, 380, 383, 386, 388, 391, 392, 393, 395, 398, 403, 405, 407, 408, 413, 415, 417, 419, 420, 422, 424, 425, 434, 437, 438, 440, 441, 442, 443, 444, 446, 449, 450, 452, 453, 456, 457, 458, 459, 461, 462, 463, 467, 469, 470, 471, 472, 473, 475, 476, 478, 479, 483, 484, 486, 487, 488, 491, 492, 493, 495, 496, 499, 500, 501, 503, 506, 507, 508, 510, 513, 514, 518, 519, 522, 523, 524, 525, 528, 533, 535, 540, 542, 543, 545, 547, 548, 553, 554, 561, 562, 567, 570, 571, 573, 574, 578, 583, 584, 585, 587, 588, 589, 590, 591, 593, 595, 596, 597, 598, 600, 602, 606, 607, 610, 612, 613, 614, 617, 619, 622, 623, 626, 629, 631, 633, 634, 635, 636, 638, 641, 644, 645, 646, 647, 649, 650, 653, 657, 659, 660, 663, 666, 668, 669, 670, 673, 675, 676, 678, 679, 681, 682, 684, 688, 690, 691, 694, 695, 696, 700, 703, 704, 706, 707, 709, 710, 711, 712, 714, 715, 717, 718, 722, 723, 724, 725, 727, 731, 735, 744, 745, 746, 748, 749, 750, 751, 752, 753, 755, 756, 760, 762, 766, 767, 770, 772, 777, 781, 782, 786, 787, 788, 790, 792, 793, 794, 796, 797, 799, 800, 801, 803, 804, 805, 808, 809, 810, 811, 812, 817, 820, 822, 823, 824, 835, 836, 837, 840, 849, 850, 854, 856, 858, 862, 866, 868, 872, 873, 874, 877, 879, 880, 882, 883, 886, 887, 888, 890, 895, 898, 900, 901, 903, 904, 907, 916, 917, 918, 919, 920, 925, 928, 929, 930, 934, 935, 936, 937]
print(rerun_v10)
print(len(rerun_v10))

[498, 7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912, 2, 4, 5, 6, 11, 13, 14, 16, 17, 19, 20, 21, 27, 28, 29, 30, 32, 33, 35, 37, 41, 43, 44, 45, 49, 51, 53, 54, 58, 61, 63, 64, 72, 73, 75, 79, 83, 88, 91, 94, 95, 97, 98, 102, 108, 111, 112, 113, 117, 120, 124, 125, 129, 131, 134, 137, 138, 141, 143, 144, 147, 148, 149, 150, 152, 153, 157, 161, 162, 163, 164, 165, 169, 170, 172, 173, 175, 177, 181, 182, 184, 188, 192, 193, 194, 195, 197, 198, 199, 200, 204, 210, 211, 213, 215, 220, 222, 223, 225, 227, 229, 232, 233, 235, 236, 241, 242, 243, 247, 248, 249, 250, 255, 256, 257, 266, 267, 269, 274, 275, 277, 281, 285, 286, 294, 296, 297, 301, 304, 308, 310, 312, 313, 316, 317, 318, 319, 320, 322, 323, 326, 329, 331, 333, 338, 339, 340, 345, 347, 353, 354, 356, 363, 367, 369, 372, 373, 377, 378, 380, 383, 386, 388, 391, 392, 393, 395, 398, 403, 405, 407, 408, 413, 415, 417, 419, 420, 422, 424, 425, 434, 437, 438

In [1]:
rerun_v11 = [498] + [7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912] + [2, 5, 11, 13, 14, 16, 17, 19, 20, 21, 28, 30, 33, 35, 37, 41, 43, 44, 45, 49, 54, 58, 63, 64, 72, 73, 79, 83, 91, 95, 98, 102, 112, 113, 117, 120, 124, 125, 137, 138, 141, 143, 144, 147, 149, 150, 152, 153, 157, 161, 162, 164, 165, 170, 173, 175, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 210, 211, 213, 220, 222, 223, 225, 229, 233, 235, 241, 242, 243, 248, 249, 250, 255, 257, 266, 269, 274, 275, 281, 285, 286, 297, 301, 308, 310, 312, 313, 316, 317, 318, 319, 322, 323, 326, 329, 331, 333, 340, 345, 347, 353, 356, 367, 373, 377, 378, 380, 383, 386, 388, 392, 398, 403, 405, 413, 415, 417, 419, 422, 424, 434, 437, 440, 442, 443, 444, 446, 449, 450, 452, 453, 456, 457, 458, 461, 462, 467, 469, 470, 471, 472, 475, 476, 478, 479, 483, 486, 487, 488, 491, 493, 495, 496, 499, 500, 501, 503, 506, 507, 508, 510, 513, 518, 522, 523, 525, 528, 533, 540, 542, 554, 562, 570, 571, 573, 574, 578, 583, 585, 587, 589, 590, 591, 593, 595, 596, 598, 600, 602, 606, 614, 617, 619, 622, 629, 631, 633, 634, 636, 638, 644, 645, 646, 647, 649, 653, 657, 659, 660, 663, 666, 668, 673, 675, 679, 682, 688, 690, 691, 694, 695, 700, 703, 704, 712, 718, 722, 724, 725, 727, 744, 746, 748, 749, 751, 753, 760, 762, 766, 772, 782, 786, 787, 792, 793, 796, 799, 800, 801, 805, 808, 809, 810, 817, 823, 824, 835, 836, 840, 850, 854, 856, 858, 866, 868, 872, 873, 874, 877, 883, 887, 901, 904, 907, 917, 919, 920, 925, 928, 929, 930, 934, 936, 937]
print(rerun_v11)
print(len(rerun_v11))

[498, 7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912, 2, 5, 11, 13, 14, 16, 17, 19, 20, 21, 28, 30, 33, 35, 37, 41, 43, 44, 45, 49, 54, 58, 63, 64, 72, 73, 79, 83, 91, 95, 98, 102, 112, 113, 117, 120, 124, 125, 137, 138, 141, 143, 144, 147, 149, 150, 152, 153, 157, 161, 162, 164, 165, 170, 173, 175, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 210, 211, 213, 220, 222, 223, 225, 229, 233, 235, 241, 242, 243, 248, 249, 250, 255, 257, 266, 269, 274, 275, 281, 285, 286, 297, 301, 308, 310, 312, 313, 316, 317, 318, 319, 322, 323, 326, 329, 331, 333, 340, 345, 347, 353, 356, 367, 373, 377, 378, 380, 383, 386, 388, 392, 398, 403, 405, 413, 415, 417, 419, 422, 424, 434, 437, 440, 442, 443, 444, 446, 449, 450, 452, 453, 456, 457, 458, 461, 462, 467, 469, 470, 471, 472, 475, 476, 478, 479, 483, 486, 487, 488, 491, 493, 495, 496, 499, 500, 501, 503, 506, 507, 508, 510, 513, 518, 522, 523, 525, 528,

In [1]:
rerun_v13 = [498] + [7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912] + [2, 5, 11, 16, 17, 19, 21, 30, 33, 37, 41, 45, 49, 54, 58, 63, 73, 83, 91, 95, 98, 102, 112, 117, 120, 124, 141, 144, 152, 153, 161, 164, 165, 170, 173, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 211, 222, 223, 225, 229, 235, 242, 243, 248, 249, 250, 255, 266, 269, 274, 275, 281, 285, 297, 308, 310, 312, 316, 322, 329, 331, 347, 353, 356, 367, 373, 377, 378, 383, 386, 388, 403, 405, 413, 419, 422, 434, 443, 446, 449, 450, 453, 456, 457, 458, 469, 470, 475, 479, 483, 486, 487, 488, 491, 493, 495, 496, 500, 501, 506, 507, 508, 518, 523, 525, 533, 540, 554, 562, 570, 571, 578, 583, 585, 587, 589, 590, 591, 593, 596, 598, 602, 606, 619, 622, 629, 631, 633, 634, 636, 638, 644, 646, 647, 649, 657, 659, 660, 668, 673, 679, 682, 688, 690, 691, 695, 700, 704, 718, 722, 724, 725, 727, 744, 748, 751, 753, 786, 787, 793, 799, 800, 801, 808, 809, 810, 823, 824, 836, 840, 850, 856, 858, 866, 868, 872, 873, 874, 883, 887, 901, 904, 907, 917, 919, 925, 928, 930, 934, 936]
print(rerun_v13)
print(len(rerun_v13))

[498, 7, 46, 68, 74, 82, 93, 187, 284, 330, 376, 409, 418, 435, 445, 468, 490, 552, 586, 620, 652, 814, 838, 842, 857, 894, 911, 912, 2, 5, 11, 16, 17, 19, 21, 30, 33, 37, 41, 45, 49, 54, 58, 63, 73, 83, 91, 95, 98, 102, 112, 117, 120, 124, 141, 144, 152, 153, 161, 164, 165, 170, 173, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 211, 222, 223, 225, 229, 235, 242, 243, 248, 249, 250, 255, 266, 269, 274, 275, 281, 285, 297, 308, 310, 312, 316, 322, 329, 331, 347, 353, 356, 367, 373, 377, 378, 383, 386, 388, 403, 405, 413, 419, 422, 434, 443, 446, 449, 450, 453, 456, 457, 458, 469, 470, 475, 479, 483, 486, 487, 488, 491, 493, 495, 496, 500, 501, 506, 507, 508, 518, 523, 525, 533, 540, 554, 562, 570, 571, 578, 583, 585, 587, 589, 590, 591, 593, 596, 598, 602, 606, 619, 622, 629, 631, 633, 634, 636, 638, 644, 646, 647, 649, 657, 659, 660, 668, 673, 679, 682, 688, 690, 691, 695, 700, 704, 718, 722, 724, 725, 727, 744, 748, 751, 753, 786, 787, 793, 799, 800, 801, 808, 809,

In [25]:
from judger import Judger
judger = Judger(strict_extract=False)

gold_list = ["77.2*exp(0.016*t)", "92.056", "2020"]
judger.auto_judge(
    # pred="\\boxed{0.785398163}",
    # pred="\\boxed{\\atan(1)}",
    pred="\\boxed{77.2\\exp{0.016t},92.053,2020}",
    gold=gold_list,
    options=[[]] * len(gold_list),
)

['77.2*\\exp^{1}(0.016*t)', '92.056', '2020']
77.2\exp^{1}{0.016t} 77.2*\exp^{1}(0.016*t
77.2\exp^{1}{0.016t} 77.2*\exp^{1}(0.016*t
77.2\exp^{1}{0.016t} 77.2*\exp^{1}(0.016*t
77.2\exp^{1}{0.016t} 77.2*\exp^{1}(0.016*t
77.2\exp^{1}{0.016t} 77.2*\exp^{1}(0.016*t)
92.053 92.056
92.053 92.056
92.053 92.056
92.053 92.056
92.053 92.056
92.053 92.056


False

In [22]:
print(len([7, 11, 19, 54, 58, 63, 91, 93, 95, 112, 144, 161, 164, 170, 192, 194, 199, 211, 222, 225, 229, 235, 242, 250, 275, 284, 285, 308, 312, 316, 331, 373, 376, 378, 383, 391, 409, 413, 422, 443, 445, 446, 450, 453, 457, 458, 487, 488, 493, 498, 501, 507, 519, 523, 552, 571, 585, 586, 589, 593, 602, 629, 646, 650, 652, 673, 682, 688, 690, 706, 722, 724, 748, 751, 786, 799, 801, 809, 814, 823, 840, 842, 856, 883, 911]))

85
